In [4]:
# Install required packages for the project
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
%pip install scikit-learn pandas numpy matplotlib seaborn umap-learn
%pip install captum plotly astropy tqdm
%pip install --upgrade --force-reinstall numpy pandas

# Optional: Install ztfquery for real ZTF data (requires IRSA account)
# pip install ztfquery

print("✅ All packages installed successfully!")

Looking in indexes: https://download.pytorch.org/whl/cpu
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 91.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 63.1 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 18.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 347.8/347.8 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: tzdata
    Found existing installation: tzdata 2025.2
    Uninstalling tzdata-2025.2:
      Successfully uninstalled tzdata-2025.2
  Attempting uninstall: six
    Found existing installatio

✅ All packages installed successfully!


In [3]:
"""
Complete ZTF Light Curve Dataset Downloader (FIXED)
Downloads multi-band light curves for rare astronomical transients
Optimized for contrastive learning research
"""

import requests
import pandas as pd
import numpy as np
import time
import json
from typing import List, Dict, Tuple
from datetime import datetime
import os

class ZTFDatasetDownloader:
    """
    Download and preprocess ZTF light curve datasets for rare transient classification
    """
    
    def __init__(self, output_dir: str = 'ztf_dataset'):
        self.base_url = "https://api.alerce.online/ztf/v1"
        self.output_dir = output_dir
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        print(f"Output directory: {output_dir}")
        
    def search_objects(self, 
                      classifier: str = None,
                      class_name: str = None,
                      page: int = 1,
                      page_size: int = 100) -> Dict:
        """
        Search for ZTF objects matching criteria
        
        Args:
            classifier: Classifier name ('lc_classifier', 'stamp_classifier')
            class_name: Transient class
            page: Page number for pagination
            page_size: Number of results per page (max 100)
        
        Returns:
            Dictionary with search results
        """
        url = f"{self.base_url}/objects"
        
        params = {
            'page': page,
            'page_size': page_size,
            'order_by': 'ndet',
            'order_mode': 'DESC'
        }
        
        # Add classifier and class filter if provided
        if classifier and class_name:
            params['classifier'] = classifier
            params['class'] = class_name
            
        try:
            response = requests.get(url, params=params, timeout=30)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")
            return {'items': [], 'total': 0}
    
    def get_classifier_classes(self) -> List[str]:
        """
        Get available classifier classes from the API
        
        Returns:
            List of available class names
        """
        url = f"{self.base_url}/classifiers"
        
        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            classifiers = response.json()
            
            # Extract class names from all classifiers
            all_classes = set()
            for classifier in classifiers:
                if 'classes' in classifier:
                    all_classes.update(classifier['classes'])
            
            return sorted(list(all_classes))
        except Exception as e:
            print(f"Error fetching classifiers: {e}")
            return []
    
    def get_object_ids(self, 
                      class_name: str = None,
                      n_objects: int = 100,
                      classifier: str = 'lc_classifier') -> List[str]:
        """
        Get list of object IDs for a specific class
        
        Args:
            class_name: Transient class name (None for all objects)
            n_objects: Number of objects to fetch
            classifier: Classifier to use
        
        Returns:
            List of ZTF object IDs
        """
        object_ids = []
        page = 1
        
        class_str = class_name if class_name else "All"
        print(f"\nSearching for {class_str} objects...")
        
        while len(object_ids) < n_objects:
            result = self.search_objects(
                classifier=classifier if class_name else None,
                class_name=class_name,
                page=page,
                page_size=100
            )
            
            items = result.get('items', [])
            if not items:
                print(f"No more objects found. Got {len(object_ids)} total.")
                break
            
            for item in items:
                oid = item.get('oid')
                ndet = item.get('ndet', 0)
                
                # Filter by minimum detections
                if oid and ndet >= 20:
                    object_ids.append(oid)
                    
                if len(object_ids) >= n_objects:
                    break
            
            print(f"  Collected {len(object_ids)}/{n_objects} objects (page {page})", end='\r')
            page += 1
            time.sleep(0.5)  # Rate limiting
            
            # Safety check - don't paginate forever
            if page > 50:
                break
        
        print(f"\nFound {len(object_ids)} {class_str} objects with sufficient detections")
        return object_ids[:n_objects]
    
    def get_light_curve(self, oid: str) -> Tuple[pd.DataFrame, Dict]:
        """
        Fetch light curve detections and metadata for an object
        
        Args:
            oid: ZTF object identifier
        
        Returns:
            Tuple of (light_curve_df, metadata_dict)
        """
        # Get detections
        det_url = f"{self.base_url}/objects/{oid}/detections"
        
        # Get metadata
        meta_url = f"{self.base_url}/objects/{oid}"
        
        try:
            # Fetch detections
            det_response = requests.get(det_url, timeout=20)
            det_response.raise_for_status()
            detections = det_response.json()
            
            # Fetch metadata
            meta_response = requests.get(meta_url, timeout=20)
            meta_response.raise_for_status()
            metadata = meta_response.json()
            
            # Process detections into DataFrame
            if detections:
                lc_data = []
                for det in detections:
                    lc_data.append({
                        'oid': oid,
                        'mjd': det.get('mjd'),
                        'fid': det.get('fid'),  # 1=g-band, 2=r-band
                        'mag': det.get('mag'),
                        'e_mag': det.get('e_mag'),
                        'magpsf': det.get('magpsf'),
                        'sigmapsf': det.get('sigmapsf'),
                        'ra': det.get('ra'),
                        'dec': det.get('dec'),
                        'isdiffpos': det.get('isdiffpos')
                    })
                
                df = pd.DataFrame(lc_data)
                
                # Filter only g and r bands (fid 1 and 2)
                df = df[df['fid'].isin([1, 2])]
                
                # Sort by time
                df = df.sort_values('mjd').reset_index(drop=True)
                
                return df, metadata
            else:
                return pd.DataFrame(), metadata
                
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {oid}: {e}")
            return pd.DataFrame(), {}
    
    def download_class_dataset(self,
                               class_name: str = None,
                               n_objects: int = 100,
                               min_detections: int = 20,
                               classifier: str = 'lc_classifier',
                               save_individual: bool = False) -> pd.DataFrame:
        """
        Download complete dataset for a transient class
        
        Args:
            class_name: Transient class name (None for diverse sample)
            n_objects: Number of objects to download
            min_detections: Minimum detections required
            classifier: Classifier to use
            save_individual: Save individual light curves to separate files
        
        Returns:
            Combined DataFrame with all light curves
        """
        class_str = class_name if class_name else "Mixed"
        print(f"\n{'='*60}")
        print(f"Downloading {class_str} dataset")
        print(f"{'='*60}")
        
        # Get object IDs
        object_ids = self.get_object_ids(class_name, n_objects, classifier)
        
        if not object_ids:
            print(f"No objects found for {class_str}")
            return pd.DataFrame()
        
        # Download light curves
        all_lcs = []
        metadata_list = []
        successful = 0
        
        print(f"\nDownloading light curves...")
        
        for i, oid in enumerate(object_ids):
            print(f"  Progress: {i+1}/{len(object_ids)} - {oid} - Success: {successful}", end='\r')
            
            lc_df, metadata = self.get_light_curve(oid)
            
            if not lc_df.empty and len(lc_df) >= min_detections:
                # Extract class from metadata
                if class_name:
                    lc_df['class'] = class_name
                else:
                    # Try to get class from metadata
                    classifiers = metadata.get('probabilities', {})
                    if classifiers:
                        lc_classifier = classifiers.get('lc_classifier', {})
                        if lc_classifier:
                            predicted_class = max(lc_classifier.items(), key=lambda x: x[1])[0]
                            lc_df['class'] = predicted_class
                        else:
                            lc_df['class'] = 'Unknown'
                    else:
                        lc_df['class'] = 'Unknown'
                
                all_lcs.append(lc_df)
                
                metadata['assigned_class'] = lc_df['class'].iloc[0]
                metadata_list.append(metadata)
                
                # Save individual light curve if requested
                if save_individual:
                    safe_class = lc_df['class'].iloc[0].replace('/', '_')
                    lc_file = os.path.join(self.output_dir, f"{safe_class}_{oid}.csv")
                    lc_df.to_csv(lc_file, index=False)
                
                successful += 1
            
            # Rate limiting
            time.sleep(0.5)
        
        print(f"\n\nSuccessfully downloaded {successful}/{len(object_ids)} light curves")
        
        # Combine all light curves
        if all_lcs:
            combined_df = pd.concat(all_lcs, ignore_index=True)
            
            # Save combined dataset
            safe_class_name = (class_name.replace('/', '_') if class_name else 'mixed')
            output_file = os.path.join(self.output_dir, f'{safe_class_name}_lightcurves.csv')
            combined_df.to_csv(output_file, index=False)
            print(f"Saved to: {output_file}")
            
            # Save metadata
            meta_file = os.path.join(self.output_dir, f'{safe_class_name}_metadata.json')
            with open(meta_file, 'w') as f:
                json.dump(metadata_list, f, indent=2)
            
            return combined_df
        else:
            return pd.DataFrame()
    
    def download_diverse_dataset(self,
                                total_objects: int = 300,
                                min_detections: int = 20) -> pd.DataFrame:
        """
        Download a diverse dataset without specifying classes
        (Lets the API return various transient types)
        
        Args:
            total_objects: Total number of objects to download
            min_detections: Minimum detections per object
        
        Returns:
            Combined DataFrame with diverse transient types
        """
        print(f"\n{'='*60}")
        print(f"DOWNLOADING DIVERSE TRANSIENT DATASET")
        print(f"{'='*60}")
        print(f"Target: {total_objects} objects with {min_detections}+ detections")
        
        dataset = self.download_class_dataset(
            class_name=None,  # Get diverse sample
            n_objects=total_objects,
            min_detections=min_detections,
            classifier=None  # Don't filter by classifier
        )
        
        if not dataset.empty:
            # Print class distribution
            print(f"\n{'='*60}")
            print("CLASS DISTRIBUTION IN DOWNLOADED DATA")
            print(f"{'='*60}")
            class_counts = dataset.groupby('class')['oid'].nunique().sort_values(ascending=False)
            for class_name, count in class_counts.items():
                print(f"  {class_name}: {count} objects")
            
            # Save summary
            output_file = os.path.join(self.output_dir, 'diverse_dataset.csv')
            dataset.to_csv(output_file, index=False)
            print(f"\nDataset saved to: {output_file}")
            
        return dataset
    
    def explore_sample_objects(self, n_samples: int = 10) -> pd.DataFrame:
        """
        Download a small sample to explore data structure
        
        Args:
            n_samples: Number of sample objects
        
        Returns:
            Sample DataFrame
        """
        print(f"Downloading {n_samples} sample objects...")
        
        # Get random objects
        result = self.search_objects(page=1, page_size=n_samples)
        items = result.get('items', [])
        
        if not items:
            print("No objects found")
            return pd.DataFrame()
        
        samples = []
        
        for i, item in enumerate(items[:n_samples]):
            oid = item.get('oid')
            print(f"  Fetching {i+1}/{n_samples}: {oid}", end='\r')
            
            lc_df, metadata = self.get_light_curve(oid)
            
            if not lc_df.empty:
                # Get class from metadata
                classifiers = metadata.get('probabilities', {})
                if classifiers:
                    lc_classifier = classifiers.get('lc_classifier', {})
                    if lc_classifier:
                        predicted_class = max(lc_classifier.items(), key=lambda x: x[1])[0]
                        lc_df['class'] = predicted_class
                
                samples.append(lc_df)
            
            time.sleep(0.5)
        
        if samples:
            sample_df = pd.concat(samples, ignore_index=True)
            print(f"\n\nSample data structure:")
            print(sample_df.head())
            print(f"\nClasses found: {sample_df['class'].unique()}")
            return sample_df
        else:
            return pd.DataFrame()


# =============================================================================
# SIMPLIFIED EXECUTION FUNCTIONS
# =============================================================================

def quick_diverse_download(n_objects: int = 100):
    """
    Quick start: Download diverse transients without specifying classes
    This is the RECOMMENDED approach for your research
    """
    print("="*60)
    print("QUICK START - DIVERSE TRANSIENT DOWNLOAD")
    print("="*60)
    print("\nThis will download objects of various types automatically")
    print("The API will return a mix of common and rare transients\n")
    
    downloader = ZTFDatasetDownloader(output_dir='ztf_diverse_data')
    
    dataset = downloader.download_diverse_dataset(
        total_objects=n_objects,
        min_detections=20
    )
    
    if not dataset.empty:
        print("\n" + "="*60)
        print("DOWNLOAD COMPLETE - SUMMARY")
        print("="*60)
        print(f"Total objects: {dataset['oid'].nunique()}")
        print(f"Total detections: {len(dataset)}")
        
        print("\nFilter distribution:")
        filter_map = {1: 'g-band', 2: 'r-band'}
        print(dataset['fid'].map(filter_map).value_counts())
        
        print("\nDetections per object:")
        dets_per_obj = dataset.groupby('oid').size()
        print(f"  Min: {dets_per_obj.min()}")
        print(f"  Max: {dets_per_obj.max()}")
        print(f"  Mean: {dets_per_obj.mean():.1f}")
        
        print("\n✓ Dataset ready for contrastive learning!")
        print(f"  File: ztf_diverse_data/diverse_dataset.csv")
        
        return dataset
    else:
        print("\nDownload failed. Check your internet connection.")
        return None


def explore_api():
    """
    Explore the API and data structure with a small sample
    """
    print("="*60)
    print("EXPLORING ZTF API AND DATA STRUCTURE")
    print("="*60)
    
    downloader = ZTFDatasetDownloader(output_dir='ztf_exploration')
    
    # Download small sample
    sample = downloader.explore_sample_objects(n_samples=5)
    
    if not sample.empty:
        print("\n✓ Sample downloaded successfully!")
        print(f"  Check: ztf_exploration/ directory")
        
        # Show structure
        print("\nDataFrame columns:")
        print(sample.columns.tolist())
        
        print("\nSample light curve (first object):")
        first_oid = sample['oid'].iloc[0]
        print(sample[sample['oid'] == first_oid][['mjd', 'fid', 'mag', 'e_mag']].head(10))
        
        return sample
    else:
        print("\nExploration failed.")
        return None


# =============================================================================
# MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    # OPTION 1: Quick diverse download (RECOMMENDED)
    # This gets you a mix of transients without worrying about class names
    
    print("\nStarting download in 3 seconds...")
    print("(This will download ~100 diverse transients)\n")
    time.sleep(3)
    
    dataset = quick_diverse_download(n_objects=100)
    
    # OPTION 2: Explore first (uncomment to use)
    # sample = explore_api()
    
    # OPTION 3: Download specific classes (if you know the exact class names)
    # downloader = ZTFDatasetDownloader(output_dir='ztf_custom')
    # sn_data = downloader.download_class_dataset(
    #     class_name='SN',  # Try simple class names
    #     n_objects=50,
    #     classifier='lc_classifier'
    # )


Starting download in 3 seconds...
(This will download ~100 diverse transients)

QUICK START - DIVERSE TRANSIENT DOWNLOAD

This will download objects of various types automatically
The API will return a mix of common and rare transients

Output directory: ztf_diverse_data

DOWNLOADING DIVERSE TRANSIENT DATASET
Target: 100 objects with 20+ detections


Searching for All objects...
  Collected 100/100 objects (page 1)
Found 100 All objects with sufficient detections

  Progress: 100/100 - ZTF18aairqhf - Success: 99

Successfully downloaded 100/100 light curves
Saved to: ztf_diverse_data\mixed_lightcurves.csv

CLASS DISTRIBUTION IN DOWNLOADED DATA
  Unknown: 100 objects

Dataset saved to: ztf_diverse_data\diverse_dataset.csv

DOWNLOAD COMPLETE - SUMMARY
Total objects: 100
Total detections: 313711

Filter distribution:
fid
r-band    172235
g-band    141476
Name: count, dtype: int64

Detections per object:
  Min: 2840
  Max: 4310
  Mean: 3137.1

✓ Dataset ready for contrastive learning!
  F

In [4]:
import pandas as pd
from typing import List, Optional

def save_first_n_rows(
    input_csv: str,
    output_csv: str,
    drop_cols: Optional[List[str]] = None,
    n: int = 20000,
    **read_csv_kwargs
) -> str:
    """
    Load a CSV file, drop selected columns, and save the first n rows.

    Parameters
    ----------
    input_csv : str
        Path to input CSV file.
    output_csv : str
        Path to save the output CSV.
    drop_cols : list[str], optional
        Columns to drop.
    n : int
        Number of rows to keep.
    read_csv_kwargs : dict
        Extra arguments for pandas.read_csv().

    Returns
    -------
    str
        Path to the saved CSV file.
    """

    df = pd.read_csv(input_csv, **read_csv_kwargs)

    if drop_cols:
        cols_to_drop = [c for c in drop_cols if c in df.columns]
        df = df.drop(columns=cols_to_drop)

    df.head(n).to_csv(output_csv, index=False)
    return output_csv


In [5]:
save_first_n_rows(
    input_csv="diverse_dataset.csv",
    output_csv="light_curves.csv",
    drop_cols=["class"],
    n=20000
)

'light_curves.csv'

In [1]:
import requests

# Example: Fetch light curve for a ZTF object
oid = "ZTF18aaaaaaa"  # Example object ID
url = f"https://api.alerce.online/ztf/v1/objects/{oid}/detections"
response = requests.get(url)
light_curve_data = response.json()

In [ ]:
import os
import requests
import pandas as pd
from io import StringIO
from dotenv import load_dotenv

load_dotenv()

USER = os.getenv("IRSA_USER")
PASS = os.getenv("IRSA_PASS")


# Step 1: Search for available ZTF images
def search_ztf_images(field=None, ra=None, dec=None):
    """Search for ZTF images"""
    search_url = "https://irsa.ipac.caltech.edu/ibe/search/ztf/products/sci"

    params = {'ct': 'csv'}

    # Build WHERE clause based on search criteria
    where_clauses = []
    if field:
        where_clauses.append(f"field={field}")
    if ra and dec:
        where_clauses.append(f"POS={ra},{dec}")

    if where_clauses:
        params['WHERE'] = " AND ".join(where_clauses)

    response = requests.get(search_url, params=params, auth=(USER, PASS))

    if response.status_code == 200:
        # Parse CSV response
        df = pd.read_csv(StringIO(response.text))
        return df
    else:
        print(f"Search failed: {response.status_code}")
        print(response.text)
        return None

# Step 2: Download a specific file
def download_ztf_file(file_path, output_filename):
    """Download a ZTF file using the file path from search results"""
    base_url = "https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"
    full_url = f"{base_url}/{file_path}"

    print(f"Downloading: {full_url}")

    response = requests.get(full_url, auth=(USER, PASS))

    if response.status_code == 200:
        with open(output_filename, 'wb') as f:
            f.write(response.content)
        print(f"Successfully downloaded: {output_filename}")
        return True
    else:
        print(f"Download failed: {response.status_code}")
        print(response.text[:200])
        return False

# Example usage
print("Searching for ZTF images...")
results = search_ztf_images(field=570)  # Search for field 570

if results is not None and len(results) > 0:
    print(f"Found {len(results)} images")
    print("\nFirst few results:")
    print(results.head())

    # Download the first image
    if 'filefracday' in results.columns:
        first_file = results.iloc[0]
        # Construct the file path from the metadata
        # This depends on the actual column names in the results
        print("\nAttempting to download first image...")
        # You'll need to construct the path based on the actual column names
        print(first_file)
else:
    print("No results found")
